<a href="https://colab.research.google.com/github/juanselles/book/blob/main/8_Hyperparameter_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# =========================================================
# 0. PREPARACIÓN DE DATOS (Necesario para que el código funcione)
# =========================================================
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# Cargamos un dataset real de Scikit-Learn (diagnóstico de cáncer)
X, y = load_breast_cancer(return_X_y=True)

# Dividimos los datos: 80% para entrenar (train) y 20% para examinar (test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



Evaluando combinaciones de hiperparámetros. Por favor, espera...

¡Búsqueda finalizada!
Configuración óptima: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
Mejor rendimiento en Train (Accuracy): 0.9626
Precisión final en Test (datos nuevos): 0.9649


In [ ]:

# =========================================================
# --- INICIO DEL CÓDIGO DEL LIBRO ---
# =========================================================
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# 1. Definición del modelo base
modelo = RandomForestClassifier(random_state=42)

# 2. Definición del diccionario de hiperparámetros (la «rejilla»)
parametros = {
    'n_estimators': [50, 100, 200],      # Número total de árboles
    'max_depth': [None, 10, 20, 30],     # Profundidad máxima permitida
    'min_samples_split': [2, 5, 10]      # Mínimo de muestras para dividir un nodo
}

# 3. Configuración de la búsqueda
# cv=5 indica que ejecutará 5 rondas de validación cruzada por combinación
busqueda = GridSearchCV(estimator=modelo, param_grid=parametros, cv=5, scoring='accuracy')

# 4. Ejecución del entrenamiento exhaustivo
print("Evaluando combinaciones de hiperparámetros. Por favor, espera...")
busqueda.fit(X_train, y_train)

# 5. Obtención de resultados
print("\n¡Búsqueda finalizada!")
print("Configuración óptima:", busqueda.best_params_)
print("Mejor rendimiento en Train (Accuracy):", round(busqueda.best_score_, 4))

# Almacenamiento del modelo optimizado para predicciones futuras
# (Nota: Se ha eliminado el punto ortográfico del final para evitar el error de sintaxis)
mejor_modelo = busqueda.best_estimator_

# =========================================================
# EXTRA: Comprobación final en el examen real (Test)
# =========================================================
precision_test = mejor_modelo.score(X_test, y_test)
print("Precisión final en Test (datos nuevos):", round(precision_test, 4))

In [3]:

# =========================================================
# EXTRA: OPTIMIZACIÓN CON XGBOOST Y RANDOMIZEDSEARCHCV
# =========================================================
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# 1. Definición del modelo XGBoost
xgb_modelo = XGBClassifier(random_state=42, eval_metric='logloss')

# 2. Definición del espacio de búsqueda aleatorio
# RandomizedSearchCV explora combinaciones de forma aleatoria,
# ideal para espacios grandes donde GridSearch tardaría horas.
distribucion_parametros = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(3, 10),
    'learning_rate': [0.01, 0.1, 0.2]
}

# 3. Configuración de RandomizedSearchCV
# n_iter=20 significa que probará 20 combinaciones aleatorias.
random_search = RandomizedSearchCV(
    estimator=xgb_modelo,
    param_distributions=distribucion_parametros,
    n_iter=20,
    cv=5,
    scoring='accuracy',
    random_state=42
)

# 4. Entrenamiento
print("\nEjecutando búsqueda aleatoria con XGBoost...")
random_search.fit(X_train, y_train)

# 5. Resultados
print("Configuración óptima (XGBoost):", random_search.best_params_)
print("Mejor rendimiento en Train (Accuracy):", round(random_search.best_score_, 4))

# 6. Comprobación final en el examen real
precision_xgb = random_search.best_estimator_.score(X_test, y_test)
print("Precisión final en Test (XGBoost):", round(precision_xgb, 4))


Ejecutando búsqueda aleatoria con XGBoost...
Configuración óptima (XGBoost): {'learning_rate': 0.1, 'max_depth': 8, 'n_estimators': 285}
Mejor rendimiento en Train (Accuracy): 0.967
Precisión final en Test (XGBoost): 0.9561
